### Importing :

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

### Data Loading

In [ ]:
df = pd.read_csv(r'data\fifa21 raw data v2.csv')
df

### Data Understanding :
## FIFA 21 Dataset – Column Dictionary

### Identity & Bio
- **ID** – Unique player ID (used by FIFA/sofifa)
- **Name** – Short display name (e.g. "L. Messi")
- **LongName** – Full name (e.g. "Lionel Messi")
- **photoUrl** – URL to player's photo
- **playerUrl** – URL to player's sofifa profile page
- **Nationality** – Player's country
- **Age** – Player's age (years)
- **Height** – Height as text, e.g. `"170cm"` → needs cleaning to numeric
- **Weight** – Weight as text, e.g. `"72kg"` → needs cleaning to numeric
- **Preferred Foot** – `Left` or `Right`

### Ratings & Potential
- **↓OVA** – Overall rating (current skill, 0–99). The "↓" is just a sort arrow from the original site — rename this column (e.g. to `OVA`) when cleaning
- **POT** – Potential rating (highest OVA the player could reach)
- **BOV** – Best Overall rating (rating in player's best position)

### Club / Contract
- **Club** – Current club
- **Contract** – Contract period as text, e.g. `"2018 ~ 2022"` → split into start/end years
- **Joined** – Date joined current club, e.g. `"Jul 10, 2018"` → parse to datetime
- **Loan Date End** – If on loan, date loan ends (missing if not on loan)

### Position
- **Positions** – All positions player can play, comma-separated (e.g. `"RW, ST, CF"`)
- **Best Position** – Single best position (e.g. `"RW"`)

### Value & Wages (all stored as text with currency symbol and K/M suffix — need cleaning)
- **Value** – Market value, e.g. `"€103.5M"`
- **Wage** – Weekly wage, e.g. `"€560K"`
- **Release Clause** – Buyout clause amount, e.g. `"€138.4M"`

### Attacking Attributes
- **Attacking** – Aggregate attacking score
- **Crossing** – Ability to cross the ball
- **Finishing** – Shot conversion ability
- **Heading Accuracy** – Accuracy of headers
- **Short Passing** – Short pass accuracy
- **Volleys** – Volley shot ability

### Skill Attributes
- **Skill** – Aggregate skill score
- **Dribbling** – Ball control while running/turning
- **Curve** – Ability to curve the ball
- **FK Accuracy** – Free-kick accuracy
- **Long Passing** – Long pass accuracy
- **Ball Control** – General ball control

### Movement Attributes
- **Movement** – Aggregate movement score
- **Acceleration** – Speed to reach top speed
- **Sprint Speed** – Top running speed
- **Agility** – Quickness of direction change
- **Reactions** – Speed of reacting to events
- **Balance** – Stability while running/turning

### Power Attributes
- **Power** – Aggregate power score
- **Shot Power** – Strength of shots
- **Jumping** – Jump height/ability
- **Stamina** – Endurance over a match
- **Strength** – Physical strength in duels
- **Long Shots** – Accuracy from distance

### Mentality Attributes
- **Mentality** – Aggregate mentality score
- **Aggression** – Intensity in challenges
- **Interceptions** – Ability to intercept passes
- **Positioning** – Positioning sense (esp. attackers)
- **Vision** – Ability to spot/play good passes
- **Penalties** – Penalty-taking ability
- **Composure** – Calmness under pressure

### Defending Attributes
- **Defending** – Aggregate defending score
- **Marking** – Ability to mark opponents
- **Standing Tackle** – Standing tackle ability
- **Sliding Tackle** – Sliding tackle ability

### Goalkeeping Attributes (near-zero for outfield players)
- **Goalkeeping** – Aggregate GK score
- **GK Diving** – Goalkeeper diving ability
- **GK Handling** – Goalkeeper handling/catching
- **GK Kicking** – Goalkeeper kicking power/accuracy
- **GK Positioning** – Goalkeeper positioning sense
- **GK Reflexes** – Goalkeeper reflex saves

### Overall Summary Stats
- **Total Stats** – Sum of all 34 individual attribute scores
- **Base Stats** – Sum of the six PAC/SHO/PAS/DRI/DEF/PHY scores below

### Star Ratings (stored as text with a star symbol, e.g. "4 ★" — strip the star and convert to int)
- **W/F** – Weak Foot rating (1–5 stars)
- **SM** – Skill Moves rating (1–5 stars)
- **IR** – International Reputation (1–5 stars)

### Work Rates (categorical: Low / Medium / High)
- **A/W** – Attacking Work Rate
- **D/W** – Defensive Work Rate

### FIFA "6 Core Stats" (used for radar charts / FUT cards)
- **PAC** – Pace
- **SHO** – Shooting
- **PAS** – Passing
- **DRI** – Dribbling
- **DEF** – Defending
- **PHY** – Physical

### Popularity
- **Hits** – Number of "hits"/views on the player's sofifa page (popularity proxy); stored as text, some values may need numeric conversion

### Data Exploring :

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().round(2)

In [ ]:
df.describe(include= ['object'])

In [ ]:
df.value_counts()

In [ ]:
df.isna().mean() 

In [ ]:
df.isna().mean() * 100

- Check in depth the Categorical Columns :

In [ ]:
cat_cols = df.select_dtypes(include= 'object').columns
cat_cols
for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

- Check in depth the Numerical Columns :

In [ ]:
num_cols = df.select_dtypes(include='number').columns
num_cols

for col in num_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

### Data Cleaning :

- Drop Columns

In [ ]:
df.drop(['ID', 'photoUrl','playerUrl'], axis= 1, inplace= True)

- Clean "Club" Column

In [ ]:
df['Club'] = df['Club'].str.strip()
df['Club'] = df['Club'].replace('No Club', np.nan)

- Extract and Clean "Contract: Column

In [ ]:
def contract_start_year(x):
    x = x.strip()
    if '~' in x:
        return int(x.split('~')[0].strip())   
    return np.nan

df['Contract_Start'] = df['Contract'].apply(contract_start_year).astype('Int64')

In [ ]:
df['Contract_Start']

In [ ]:
def contract_end_year(x):
    x = x.strip()
    if '~' in x:

           return ( x.split('~')[1].strip())
    return np.nan

df['Contract_End'] = df['Contract'].apply(contract_end_year).astype('Int64')

In [ ]:
df['Contract_End']

- Drop "Contract" Column

In [ ]:
df.drop(['Contract'], axis= 1, inplace= True)

- Drop "Loan Date" Column

In [ ]:
df.drop(['Loan Date End'], axis= 1, inplace= True)

- Clean "Height" Column

In [ ]:
df['Height']

In [ ]:
df['Height'] = df['Height'].str.strip('cm')
df['Height']

- Clean "Weight" Column

In [ ]:
df['Weight']

In [ ]:
df['Weight'] = df['Weight'].str.strip('kg')
df['Weight']

- Rename "OVA" Column

In [ ]:
df.rename(columns={'↓OVA': 'OVA'}, inplace=True)

- Clean "Wage" Column

In [ ]:
df['Wage']

In [ ]:
df['Wage'] = df['Wage'].str.replace('€', '').str.replace('K', '').str.strip()

In [ ]:
df['Wage'] 

- Clean "Value" Column

In [ ]:
df['Value'] = df['Value'].str.replace('€', '').str.replace('M', '').str.replace('K', '').str.strip()

In [ ]:
df['Value']

- Clean "Release Clause" Column

In [ ]:
df['Release Clause'] = df['Release Clause'].str.replace('€', '').str.replace('M', '').str.replace('K', '').str.strip()

In [ ]:
df['Release Clause']

- Clean "W/F" Column

In [ ]:
df['W/F']

In [ ]:
df['W/F'] = df['W/F'].str.strip('★')

In [ ]:
df['W/F']

- Clean "SM" Column

In [ ]:
df['SM']

In [ ]:
df['SM'] = df['SM'].str.strip('★')

In [ ]:
df['SM']

- Clean "IR" Column

In [ ]:
df['IR']

In [ ]:
df['IR'] = df['IR'].str.strip('★')

In [ ]:
df['IR']

- Clean "Joined" Column

In [ ]:
df['Joined']

In [ ]:
df['Joined'] = pd.to_datetime(df['Joined'], format='%b %d, %Y')

In [ ]:
df['Joined']

- Extract New Column to Calculate the difference Between "OVA" and "POT" :

In [ ]:
def difference_OVA_POT(row):
    return row['POT'] - row['OVA']

df['difference_OVA_POT'] = df[['POT', 'OVA']].apply(difference_OVA_POT, axis=1)

In [ ]:
df['difference_OVA_POT'] 

### Data Visualization For Numerical Columns :

In [ ]:
num_cols = df.select_dtypes(include= 'number').columns
num_cols

In [ ]:
for col in num_cols:
    px.histogram(data_frame= df, x= col, title= col).show()
    px.box(data_frame= df, x= col, title= col).show()

### Data Visualization For Categorical Columns :

In [ ]:
cat_cols = df.select_dtypes(include= 'object').columns
cat_cols

In [ ]:
for col in cat_cols:
    px.histogram(data_frame= df, x= col, title= col).show()
    

###  Basic Data Analysis and Visualization :



- How many players are in the dataset?
- How many columns are there?
- What are the average, minimum, and maximum Age?
- What is the average OVA rating?
- What is the highest-rated player?
- What is the youngest player?
- What is the oldest player?
- How many unique nationalities are there?
- How many unique clubs are there?
- What are the top 10 most common nationalities?
- What are the top 10 clubs with the most players?
- What is the average potential (POT)?
- How many players have OVA >= 80?
- How many players have POT >= 90?

In [ ]:
df

- How many players are in the dataset?


In [ ]:
df['Name'].count()

- How many columns are there?


In [ ]:
df.columns.value_counts()

- What are the average, minimum, and maximum Age?


In [ ]:
Average_ages = df['Age'].mean().round(2)
Min_age = df['Age'].min()
Max_age = df['Age'].max()

print("Average Age = ", Average_ages)
print("Maximum Age = ",Max_age)
print("Minimum Age = ",Min_age)

- What is the average OVA rating?


In [ ]:
Average_OVA = df['OVA'].mean().round(2)
print("Average Ova = ", Average_OVA)

- What is the highest-rated player?


In [ ]:
df.sort_values('OVA', ascending=False)[['Name', 'OVA', 'Club', 'Best Position']].head(1)

- What is the youngest player?


In [ ]:
df.sort_values('Age', ascending=True)[['Name', 'Age', 'Club', 'Best Position']].head(1)

- What is the oldest player?


In [ ]:
df.sort_values('Age', ascending=False)[['Name', 'Age', 'Club', 'Best Position']].head(1)

- How many unique nationalities are there?


In [ ]:
df['Nationality'].nunique()

- How many unique clubs are there?


In [ ]:
df['Club'].nunique()

- What are the top 10 most common nationalities?


In [ ]:
top10_nationalities = df['Nationality'].value_counts().head(10)
top10_nationalities

- What are the top 10 clubs with the most players?


In [ ]:
top10_clubs = df[df['Club'] != 'No Club']['Club'].value_counts().head(10)
top10_clubs

- What is the average potential (POT)?


In [ ]:
average_potential = df['POT'].mean().round(2)
average_potential

- How many players have OVA >= 80?


In [ ]:
df[df['OVA'] >= 80].value_counts().count()

- How many players have POT >= 90?

In [ ]:
df[df['POT'] >= 90].value_counts().count()

### Advanced Analysis and Visualization :


- What are the top 10 highest-rated players?
- What are the top 10 players with the highest potential?
- Which players have OVA >= 85 and POT >= 90?
- Which players have POT higher than OVA?
- Which players have the largest difference between POT and OVA?
- What is the average OVA by nationality?
- What is the average OVA by club?
- Which nationality has the highest average OVA?
- Which club has the highest average OVA?
- What is the average age by preferred foot?
- How many players use the left foot vs right foot?
- What is the average OVA for left-footed vs right-footed players?
- What is the average wage by club?
- Which clubs have the highest average player value?
- Which clubs have the highest average potential?
- What are the top 10 most valuable players?
- What are the top 10 highest-paid players?
- Which clubs spend the most on player wages?
- Which nationalities have the highest total player value?
- Which clubs have the highest combined OVA?

- What are the top 10 highest-rated players?

In [ ]:
Top10_rated_players = df.sort_values('OVA', ascending=False)[['Name', 'OVA','Age', 'Club', 'Best Position']].head(10)
Top10_rated_players

In [ ]:
fig = px.bar(
    Top10_rated_players,
    x='Name',
    y='OVA',
    color='Best Position',
    title='Top 10 Highest-Rated Players'
)
fig.show()

- What are the top 10 players with the highest potential?


In [ ]:
Top10_Potential_players = df.sort_values('POT', ascending=False)[['Name', 'POT','Age', 'Club','Best Position']].head(10)
Top10_Potential_players

In [ ]:
fig = px.bar(
    Top10_Potential_players,
    x='Name',
    y='POT',
    color='Best Position',
    title='Top 10 Highest-Rated Players'
)
fig.show()

- Which players have OVA >= 85 and POT >= 90?


In [ ]:
OVA_85_AND_POT_90 = df[(df['OVA']>=85) & (df['POT']>=90)]
OVA_85_AND_POT_90

In [ ]:
fig = px.histogram(
    OVA_85_AND_POT_90,
    x='Name',
    y='POT',
    color='Best Position',
    title='Players (OVA >= 85) AND (POT >= 90)'
)
fig.show()

- Which players have POT higher than OVA?


In [ ]:
POT_higher_than_OVA= df[(df['OVA']) < (df['POT'])]
POT_higher_than_OVA

In [ ]:
fig = px.scatter(
    POT_higher_than_OVA,
    x='OVA',
    y='POT',
    hover_name='Name',
    opacity=0.3,
    title='Players with Growth Potential (POT > OVA)'
)

fig.show()

- Which players have the largest difference between POT and OVA?


In [ ]:
difference_OVA_and_POT = df.sort_values('difference_OVA_POT', ascending=False)[['Name', 'POT', 'OVA', 'difference_OVA_POT']].head(1)
difference_OVA_and_POT

In [ ]:
fig = px.bar(
    difference_OVA_and_POT,
    x='Name',
    y='difference_OVA_POT',
    title='Player with the Largest Potential − Overall Gap',
    labels={'difference_OVA_POT': 'POT − OVA'}
)
fig.show()

- What is the average OVA by nationality?


In [ ]:
average_ova_nationality = (
    df.groupby('Nationality')['OVA']
      .mean()
      .sort_values(ascending=False)
      .reset_index().round(2).head(10)
)
average_ova_nationality

In [ ]:
fig = px.bar(
    average_ova_nationality,
    x='Nationality',
    y='OVA',
    color='Nationality',
    color_continuous_scale='Blues',
    title='Top 10 Nationalities by Average OVA'
)
fig.show()

- What is the average OVA by club?


In [ ]:
average_OVA_Club = df.groupby('Club')['OVA'].mean().sort_values(ascending=False).reset_index().round(2).head(10)
average_OVA_Club          

In [ ]:
fig = (
    px.bar(
        average_OVA_Club,
        x = 'Club',
        y = 'OVA',
        color = 'Club',
         title='Top 10 Clubs by Average OVA'

    )
)
fig.show()

- Which nationality has the highest average OVA?


In [ ]:
average_ova_nationality = (
    df.groupby('Nationality')['OVA']
      .mean()
      .sort_values(ascending=False)
      .reset_index().round(2).head(1)
)
average_ova_nationality

In [ ]:
fig = px.bar(
    average_ova_nationality,
    x='Nationality',
    y='OVA',
    color='Nationality',
    color_continuous_scale='Blues',
    title='Highest Nationality by Average OVA'
)
fig.show()

- Which club has the highest average OVA?


In [ ]:
average_OVA_Club = df.groupby('Club')['OVA'].mean().sort_values(ascending=False).reset_index().round(2).head(10)
average_OVA_Club         

In [ ]:
fig = (
    px.bar(
        average_OVA_Club,
        x = 'Club',
        y = 'OVA',
        color = 'Club',
         title='Top 10 Clubs by Average OVA'

    )
)
fig.show()

- What is the average age by preferred foot?


In [ ]:
average_Age_Foot = df.groupby('Preferred Foot')['Age'].mean().sort_values(ascending=False).reset_index().round(2).head(10)
average_Age_Foot   

In [ ]:
fig = px.bar(
    average_Age_Foot ,
    x='Preferred Foot',
    y='Age',
    title='Average Age by Preferred Foot',
    labels={
        'Age': 'Average Age',
        'Preferred Foot': 'Preferred Foot'
    }
)


fig.show()

- How many players use the left foot vs right foot?


In [ ]:
left_vs_right_foot = (
    df.groupby('Preferred Foot')['Name']
      .count()
      .sort_values(ascending=False)
      .reset_index()
)

left_vs_right_foot

In [ ]:
fig = px.histogram (
    left_vs_right_foot,
    x =  'Preferred Foot',
    y = 'Name',
    title= 'Left Foot vs Right Foot' ,
    labels= { 'Name' : 'players'}
)
fig.show()

- What is the average OVA for left-footed vs right-footed players?


In [ ]:
average_ova_preffered_foot = df.groupby('Preferred Foot')['OVA'].mean().sort_values(ascending=False).reset_index().round(2)
average_ova_preffered_foot

In [ ]:
fig = px.bar(
    average_ova_preffered_foot,
    x='Preferred Foot',
    y='OVA',
    title='Average OVA by Preferred Foot',
    labels={
        'OVA': 'Average OVA',
        'Preferred Foot': 'Preferred Foot'
    }
)

fig.show()

- What is the average wage by club?


In [ ]:
df['Wage'] = df['Wage'].astype('Int64')

In [ ]:
average_wage_club = (
    df.groupby('Club')['Wage']
      .mean()
      .sort_values(ascending=False)
      .reset_index()
      .round(2)
      .head(10)
)

average_wage_club

In [ ]:
fig = px.bar(
    average_wage_club,
    x= 'Club',
    y= 'Wage',
    title='Average Wage by Club',
    labels={
        'Wage': 'Average Wage',
        'Club': 'Club'
    }
)

fig.show()

- Which clubs have the highest average player value?

In [ ]:
df['Value'] = df['Value'].astype(float)


In [ ]:
Club_average_player_value= df.groupby('Club')['Value'].mean().sort_values(ascending=False).reset_index().round(2).head(5)
Club_average_player_value

In [ ]:
fig = px.bar(
    Club_average_player_value,
    x= 'Club',
    y= 'Value',
    title='clubs have the highest average player value',
    labels={
        'Value': 'Player Values',
        'Club': 'Top Clubs'
    }
)

fig.show()

- What are the top 10 most valuable players?


In [ ]:
top10_value_player = (
    df.sort_values(by='Value', ascending=False)
      [['Name', 'Value', 'OVA']]
      .reset_index(drop=True)
      .head(10)
)

top10_value_player


In [ ]:
fig = px.bar(
    top10_value_player,
    x='Value',
    y='Name',
    orientation='h',
    title='Top 10 Most Valuable Players',
    labels={
        'Value': 'Player Value',
        'Name': 'Player'
    }
)


fig.show()

- What are the top 10 highest-paid players?


In [ ]:
top10_highest_paid = (
    df.sort_values(by='Wage', ascending=False)
      [['Name', 'Wage', 'OVA']]
      .reset_index(drop=True)
      .head(10)
)

top10_highest_paid


In [ ]:


fig = px.bar(
    top10_highest_paid,
    x='Wage',
    y='Name',
    orientation='h',
    title='Top 10 Highest-Paid Players',
    labels={
        'Wage': 'Wage',
        'Name': 'Player'
    },
    text='Wage'
)

fig.show()


- Which clubs spend the most on player wages?


In [ ]:
top10_club_wage = (
    df.groupby('Club')['Wage']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .head(10)
)

top10_club_wage


In [ ]:


fig = px.bar(
    top10_club_wage,
    x='Wage',
    y='Club',
    orientation='h',
    title='Top 10 Clubs by Total Player Wages',
    labels={
        'Wage': 'Total Player Wages',
        'Club': 'Club'
    },
    text='Wage'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()


- Which nationalities have the highest total player value?



In [ ]:
top10_nationality_value = (
    df.groupby('Nationality')['Value']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .head(10)
)

top10_nationality_value


In [ ]:


fig = px.bar(
    top10_nationality_value,
    x='Value',
    y='Nationality',
    orientation='h',
    title='Top 10 Nationalities by Total Player Value',
    labels={
        'Value': 'Total Player Value',
        'Nationality': 'Nationality'
    },
    text='Value'
)




fig.show()


- Which clubs have the highest combined OVA?

In [ ]:
top10_club_ova = (
    df.groupby('Club')['OVA']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .head(10)
)

top10_club_ova


In [ ]:

fig = px.bar(
    top10_club_ova,
    x='OVA',
    y='Club',
    orientation='h',
    title='Top 10 Clubs by Combined OVA',
    labels={
        'OVA': 'Combined OVA',
        'Club': 'Club'
    },
    text='OVA'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()


### Correlation and Relationship between Columns :


- Does age have a relationship with overall rating?
- Does potential decrease as age increases?
- Does player wage increase with overall rating?
- Does player value increase with overall rating?

- Does age have a relationship with overall rating?


In [ ]:
corr_age_OVA = df[['Age', 'OVA']].corr()
corr_age_OVA

In [ ]:
px.imshow(corr_age_OVA, text_auto= True)


- Does potential decrease as age increases?


In [ ]:
corr_age_POT = df[['Age', 'POT']].corr()
corr_age_POT

In [ ]:
px.imshow(corr_age_OVA, text_auto= True)

- Does player wage increase with overall rating?


In [ ]:

corr_Wage_OVA = df[['Wage', 'OVA']].corr()
corr_Wage_OVA

In [ ]:
px.imshow(corr_Wage_OVA, text_auto= True)



- Does player value increase with overall rating?


In [ ]:


corr_Value_OVA = df[['Value', 'OVA']].corr()
corr_Value_OVA

In [ ]:
px.imshow(corr_Value_OVA, text_auto= True)

- Which attributes have the strongest relationship with OVA?


In [ ]:
correlation_ova = df.corr(numeric_only=True)['OVA'].sort_values(ascending=False).round(2)
correlation_ova 

In [ ]:
fig = px.bar(
    x=correlation_ova.values,
    y=correlation_ova.index,
    orientation='h',
    title='Top 10 Attributes with the Strongest Relationship with OVA',
    labels={
        'x': 'Absolute Correlation',
        'y': 'Attribute'
    },
    color=correlation_ova.values,
    color_continuous_scale='Blues'
)

fig.show()